In [3]:
# install dependencies
! pip3 install numpy

You should consider upgrading via the '/Users/wahid/Desktop/projects/small-llm/.venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
# import necessary module
import numpy as np
from urllib.request import urlopen
from io import BytesIO
import os
import math

In [79]:
def load_model():
    model_path = './models/tiny_english_gpt.npz'
    if os.path.exists(model_path):
        return np.load(model_path, allow_pickle=True)
    
    url = f'https://algo.monster/{model_path}'
    with urlopen(url) as response:
        data = response.read()
        
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        with open(model_path, 'wb') as f:
            f.write(data)

        return np.load(model_path, allow_pickle=True)

def softmax(x):
    # Apply along the last axis (rows)
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

def gelu(x):
    """GELU: smooth non-linearity"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def layer_norm(x, eps=1e-5):
    """FIXED: Normalized across the embedding features (last axis)"""
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(variance + eps)


In [93]:
model = load_model()
print([k for k in model.keys()])

# load the weights
token_embed_weight = model['token_embed.weight']
pos_embed_weight = model['pos_embed.weight']
w_q_0_w = model['blocks.0.attn.W_q.weight']
w_k_0_w = model['blocks.0.attn.W_k.weight']
w_v_0_w = model['blocks.0.attn.W_v.weight']
w_o_0_w = model['blocks.0.attn.W_o.weight']

w_q_0_b = model['blocks.0.attn.W_q.bias']
w_k_0_b = model['blocks.0.attn.W_k.bias']
w_v_0_b = model['blocks.0.attn.W_v.bias']
w_o_0_b = model['blocks.0.attn.W_o.bias']


ff_line_1_w = model['blocks.0.ff.linear1.weight']
ff_line_2_w = model['blocks.0.ff.linear2.weight']

ff_line_1_b = model['blocks.0.ff.linear1.bias']
ff_line_2_b = model['blocks.0.ff.linear2.bias']

norm_1_w = model['blocks.0.norm1.weight']
norm_2_w = model['blocks.0.norm2.weight']

norm_1_b = model['blocks.0.norm1.bias']
norm_2_b = model['blocks.0.norm2.bias']

lm_head_w = model['lm_head.weight']

vocab = model['vocab'].tolist()

['token_embed.weight', 'pos_embed.weight', 'blocks.0.attn.W_q.weight', 'blocks.0.attn.W_q.bias', 'blocks.0.attn.W_k.weight', 'blocks.0.attn.W_k.bias', 'blocks.0.attn.W_v.weight', 'blocks.0.attn.W_v.bias', 'blocks.0.attn.W_o.weight', 'blocks.0.attn.W_o.bias', 'blocks.0.ff.linear1.weight', 'blocks.0.ff.linear1.bias', 'blocks.0.ff.linear2.weight', 'blocks.0.ff.linear2.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'ln_f.weight', 'ln_f.bias', 'lm_head.weight', 'vocab', 'd_model', 'n_heads', 'n_layers', 'd_ff', 'max_seq_len']


In [67]:
inp_words = ['the', 'cat', 'sat']

In [151]:
def process(words: list[str], temp = 1.0):
    if len(words) > 10: 
        return words
        
    inp_embed = [token_embed_weight[vocab.index(word)] for word in words]

    # add position embedding
    inp_pos_embed = []
    for i in range(len(inp_embed)):
        inp_pos_embed.append(inp_embed[i] + pos_embed_weight[i])

    # Convert to a proper numpy matrix for mathematical operations
    inp_matrix = np.array(inp_pos_embed)
    inp_array = layer_norm(inp_matrix)

    Q = inp_array @ w_q_0_w + w_q_0_b
    K = inp_array @ w_k_0_w + w_k_0_b
    V = inp_array @ w_v_0_w + w_v_0_b

    dk = w_k_0_w.shape[0]
    A = Q @ K.T / math.sqrt(dk)

    seq_len = len(words)
    mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)

    A_w = np.where(mask, float('-inf'), A)
    A_w = softmax(A_w)  # Correctly placed softmax!

    O = A_w @ V

    O_projected = O @ w_o_0_w + w_o_0_b 
    
    # FIX: Using numpy matrices for the residual connection addition
    attn_out = layer_norm(inp_matrix + O_projected) 

    # Pass it to feed-forward neural network
    hidden = attn_out @ ff_line_1_w.T + ff_line_1_b
    activated = gelu(hidden)
    
    # FIX: Pass the 'activated' tensor into line 2, not 'hidden'
    ff_out_raw = activated @ ff_line_2_w.T + ff_line_2_b
    ff_output = layer_norm(attn_out + ff_out_raw) 

    # Generate prediction matrix using lm_head
    prediction_matrix = ff_output @ lm_head_w.T
    
    # Extract logits of the ABSOLUTE LAST TOKEN generated
    last_token_logits = prediction_matrix[-1]

    # Add temp scaling
    prediction = softmax(last_token_logits / temp)

    max_args = np.argsort(prediction)[::-1]

    k = 5
    top_k = max_args[:k]

    next_word = vocab[np.random.choice(top_k)]

    if next_word != 'END':
        # FIX: Explicitly forward the temperature variable parameter
        return process([*words, next_word], temp=temp)
    return words


In [152]:
res = process(inp_words, temp = 0.5)
print(' '.join(res))

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 128 is different from 64)